推荐学习资料：
https://distill.pub/2021/gnn-intro/


作业：用 GNN 预测魔法药剂的稳定性
1. 作业背景
在一个炼金术世界中，每瓶魔法药剂都由若干个“元素原子”组成。不同元素原子之间会通过“魔法键”连接，形成一张分子图。

一瓶药剂可以表示为一张图：

节点 node：元素原子
边 edge：魔法键
节点特征 node feature：元素类型
图标签 graph label：药剂是否稳定
炼金术士通过大量实验发现：
如果一瓶药剂中存在一个特殊的三角结构，那么它通常是稳定的。

这个特殊三角结构由三种元素组成：

fire, water, crystal
并且这三个元素必须两两相连，形成一个三角环：

fire
 /  \
water -- crystal
因此，本作业的目标是训练一个小型图神经网络，判断一瓶药剂是否稳定。

2. 任务目标
请使用 PyTorch 实现并训练一个简单的 GNN 模型，完成二分类任务：

输入：一张药剂图
输出：该药剂是否稳定
标签定义如下：

label = 1：图中存在 fire-water-crystal 三角结构，药剂稳定
label = 0：图中不存在该三角结构，药剂不稳定
模型需要根据节点特征和图结构，学习判断整张图的类别。

3. 数据表示
每个样本是一张固定大小的图，包含：

N_NODES = 12
也就是说，每瓶药剂由 12 个元素原子组成。

每个节点属于 5 种元素类型之一：

N_TYPES = 5
在代码中，这些类型可以理解为：

0: fire
1: water
2: crystal
3: shadow
4: wind
每个节点的元素类型会被表示成 one-hot 向量。

例如，如果某个节点是 fire，它的节点特征为：

[1, 0, 0, 0, 0]
如果某个节点是 crystal，它的节点特征为：

[0, 0, 1, 0, 0]
因此，一张图的节点特征矩阵为：

X shape = [N_NODES, N_TYPES]
在本作业中：

X shape = [12, 5]
4. 图结构表示
图结构使用邻接矩阵表示。

邻接矩阵 A 的大小为：

A shape = [N_NODES, N_NODES]
在本作业中：

A shape = [12, 12]
其中：

A[i, j] = 1 表示节点 i 和节点 j 之间有魔法键
A[i, j] = 0 表示节点 i 和节点 j 之间没有魔法键
因为魔法键没有方向，所以这是一个无向图：

A[i, j] = A[j, i]
每张图中大约有：

N_EDGES = 18条边。



流程图：
节点特征 X + 邻接矩阵 A
        ↓
邻接矩阵归一化
        ↓
第一层 GCN：节点从邻居接收信息
        ↓
ReLU
        ↓
第二层 GCN：节点继续接收更远邻居的信息
        ↓
ReLU
        ↓
对所有节点表示取平均，得到整图表示
        ↓
线性分类器
        ↓
输出稳定 / 不稳定


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(7)

In [ ]:
N_NODES = 12
N_TYPES = 5
N_EDGES = 18
HIDDEN = 32
BATCH_SIZE = 64
EPOCHS = 40

def add_edge(adj, i, j):
    adj[i, j] = 1
    adj[j, i] = 1
def count_edges(adj):
    return int(adj.sum().item() // 2)
def fill_random_edges(adj, target_edges):
    while count_edges(adj) < target_edges:
        i, j = torch.randint(0, N_NODES, (2,)).tolist()
        if i != j:
            add_edge(adj, i, j)


def has_magic_triangle(adj, types):
    # Stable potion rule:
    # A triangle containing node types {0, 1, 2}.
    for i in range(N_NODES):
        for j in range(i + 1, N_NODES):
            if adj[i, j].item() == 0:
                continue
            for k in range(j + 1, N_NODES):
                is_triangle = (
                    adj[i, k].item() == 1 and
                    adj[j, k].item() == 1
                )
                if not is_triangle:
                    continue
                tri_types = {int(types[i]), int(types[j]), int(types[k])}
                if tri_types == {0, 1, 2}:
                    return True
    return False


def make_graph(label):
    adj = torch.zeros(N_NODES, N_NODES)
    types = torch.randint(0, N_TYPES, (N_NODES,))

    if label == 1:
        # Plant one magic triangle.
        tri = torch.randperm(N_NODES)[:3]
        types[tri] = torch.tensor([0, 1, 2])
        add_edge(adj, tri[0], tri[1])
        add_edge(adj, tri[1], tri[2])
        add_edge(adj, tri[0], tri[2])
        fill_random_edges(adj, N_EDGES)
    else:
        # Generate a graph without the magic triangle.
        while True:
            adj = torch.zeros(N_NODES, N_NODES)
            types = torch.randint(0, N_TYPES, (N_NODES,))
            fill_random_edges(adj, N_EDGES)
            if not has_magic_triangle(adj, types):
                break

    x = F.one_hot(types, num_classes=N_TYPES).float()
    y = torch.tensor(label, dtype=torch.long)
    return adj.float(), x, y


class PotionDataset(Dataset):
    def __init__(self, size):
        self.data = [make_graph(i % 2) for i in range(size)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def normalize_adj(adj):
    # adj: [batch, nodes, nodes]
    batch, n, _ = adj.shape
    eye = torch.eye(n, device=adj.device).unsqueeze(0).expand(batch, n, n)
    adj = adj + eye

    deg = adj.sum(dim=-1).clamp(min=1)
    deg_inv_sqrt = deg.pow(-0.5)

    return deg_inv_sqrt.unsqueeze(2) * adj * deg_inv_sqrt.unsqueeze(1)


class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj_norm):
        # Message passing: aggregate neighbor features with adj_norm @ x.
        return torch.bmm(adj_norm, self.linear(x))


class TinyGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.gcn1 = GCNLayer(N_TYPES, HIDDEN)
        self.gcn2 = GCNLayer(HIDDEN, HIDDEN)
        self.classifier = nn.Linear(HIDDEN, 2)

    def forward(self, x, adj):
        adj_norm = normalize_adj(adj)

        h = F.relu(self.gcn1(x, adj_norm))
        h = F.relu(self.gcn2(h, adj_norm))

        graph_emb = h.mean(dim=1)
        return self.classifier(graph_emb)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for adj, x, y in loader:
        adj, x, y = adj.to(device), x.to(device), y.to(device)
        pred = model(x, adj).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    return correct / total


def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_set = PotionDataset(1200)
    test_set = PotionDataset(300)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

    model = TinyGNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0

        for adj, x, y in train_loader:
            adj, x, y = adj.to(device), x.to(device), y.to(device)

            logits = model(x, adj)
            loss = F.cross_entropy(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        if epoch % 5 == 0:
            train_loss = total_loss / len(train_set)
            test_acc = evaluate(model, test_loader, device)
            print(f"epoch {epoch:02d} | loss {train_loss:.4f} | test acc {test_acc:.3f}")

    return model


model = train()

生成数据部分代码：

In [ ]:
def addedges(A,i,j):
    #A邻接矩阵，i，j为nodes编号,对应矩阵行列，因为无向图，是symmetric,所以“加边”执行下面操作
    if i!=j:
      A[i,j]=1
      A[j,i]=1
    else:
      pass
    
def countedges(A):#数有多少个1，无向图结果为一半
    count=(A.sum().item())//2 #.item把tensor变成数值
    return count

def fill_random_edges(adj, target_edges):#加边（图生成）
    while countedges(adj) < target_edges:
        i, j = torch.randint(0, N_NODES, (2,)).tolist()#返回值是列表，长度2
        if i != j:
            addedges(adj, i, j)

def has_magic_triangle(A,types):
     for m in range(0,N_NODES):
        for n in range(0,N_NODES):
           if A[m,n].item()==0:
               continue
        for k in range(n+1,N_NODES):
           if A[m, k].item() == 1 and A[n, k].item() == 1:
                    tri_types = {int(types[m]), int(types[n]), int(types[k])}
                    if tri_types == {0, 1, 2}: #用集合判断无需顺序
                        return True
     return False

           
def make_graph(label):
    adj = torch.zeros(N_NODES, N_NODES)
    #初始化，18x18 zero matrix
    types = torch.randint(0, N_TYPES, (N_NODES,))
    #随机生成1x18tensor，每个数字(0~4)代表节点种类
    if label==1:#有特征三角
        tri = torch.randperm(N_NODES)[:3]
        types[tri] = torch.tensor([0, 1, 2])
        addedges(adj, tri[0], tri[1])
        addedges(adj, tri[1], tri[2])
        addedges(adj, tri[0], tri[2])
        fill_random_edges(adj, N_EDGES)
    else:
        # Generate a graph without the magic triangle.
        while True:
            adj = torch.zeros(N_NODES, N_NODES)
            types = torch.randint(0, N_TYPES, (N_NODES,))
            fill_random_edges(adj, N_EDGES)
            if not has_magic_triangle(adj, types):
                break

    x = F.one_hot(types, num_classes=N_TYPES).float()
    y = torch.tensor(label, dtype=torch.long)
    return adj.float(), x, y
class PotionDataset(Dataset):
    def __init__(self, size):
        self.data = [make_graph(i % 2) for i in range(size)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
    

def normalize_adj(adj):#对称归一化，适用于无向图 
   # D^(-1/2) (A + I) D^(-1/2), for adj: [batch, nodes, nodes]
    batch, n, _ = adj.shape
    I = torch.eye(n, dtype=adj.dtype, device=adj.device).expand(batch, n, n)
    A_hat = adj + I

    D_inv_sqrt = torch.diag_embed(A_hat.sum(dim=-1).clamp(min=1).pow(-0.5))
    return D_inv_sqrt @ A_hat @ D_inv_sqrt


_IncompleteInputError: incomplete input (1114015831.py, line 10)

到这里我很疑惑，为什么他的归一化要这么做

Q:请给出我能看懂的邻接矩阵归一化推导过程，并简明指出其和一般batch normalization的区别

A:邻接矩阵归一化不是为了让“均值=0、方差=1”，而是为了让每层节点聚合邻居信息时，数值不要因为邻居数量不同而变得太大或太小。
ok我看懂了，数学过程略